# Manual Event Annotation & Extraction Evaluation

Blind manual annotation of event triggers and MAVEN event types over the TMA test corpus (`experiment.jsonl`, 5,666 summaries), and MAVEN-style **trigger-identification** and **classification** F1 for the BERT+CRF extractor

## Cells (run top-to-bottom)
1. **Setup**: imports, paths, constants.
2. **Slim corpus cache**: compress the 4 GB `experiment.jsonl` to a small `gold/annotation_corpus.jsonl`.
3. **Storage helpers**: to make annotation resumable across different notebook sessions, and across different annotators
4. **Candidate selection**: deterministic seeded sample of 5 to 10 sentence summaries. Resumes where you left off.
5. **Types + FrameNet definitions**: the 168 MAVEN types with their FrameNet definition.
6. **Annotation widget**: In summary: pick sentence → trigger word(s) → search/scroll the 168 types → *Add*. (The FrameNet definition of the highlighted type is shown)
7. **Evaluation**: trigger-ID + classification P/R/F1 (exact + overlap), per-type + confusion, saved in `gold/annotation_eval.<variant>.yaml`.
8. **Inspection**: qualitative spot check to see if the annotation works on random examples
9. **Thesis table**: extraction-quality table (identification + classification, micro + macro, exact + overlap) next to the reported MAVEN reference, saved in `gold/thesis_extraction_metrics.<variant>.csv`.

## How to annotate
- Mark *every* event trigger in each summary before *Next →* (unmarked real events count as model false positives and wreck precision).
- The widget never shows the model's predictions.
- Autosaves to `gold/manual_events.<variant>.jsonl`; only summaries with ≥1 event are stored.

## Resume / extend / pseudonymized variant
- **Resume**: just re-run top-to-bottom in any later session, summaries that are annotated are skipped (though you can go "back" if the last annotation was half-finished).
- **Annotate more**: raise `N_TARGET`.
- **Pseudonymized variant**: set `VARIANT="anon"`, point `EXPERIMENT_DIR` at the pseudonymized run, and pass the cross-corpus id intersection to `select_candidates(restrict_to_ids=…)`.

In [1]:
# Imports
import json, random, collections, re, sys
from datetime import datetime
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown
import yaml
import pandas as pd

# define the root paths
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "models" / "bert_crf"))   # for utils_maven.get_labels

# Register important information: The annotator and the dataset arm (anon vs non-anon) though it's pseudonmyzed actually. 
EXPERIMENT_DIR = ROOT / "data" / "experiments" / "experiment_test_9385_20260609_1013"
VARIANT   = "non_anon"          # "non_anon" | "anon"
ANNOTATOR = "bugra"             # Annotator name, if other name is entered, new data will be saved
SEED      = 20260609            # Seeds the randomness of annotation-ready summaries. Keep it for determinism and reproduciblilty. 
MIN_SENT, MAX_SENT = 5, 10      # Change this to tweak the minimum and maximum num of sentences each annotation-ready summary will have
N_TARGET  = 30                  # raise later to annotate more; continues the same order

# Define the paths
CORPUS_PATH   = EXPERIMENT_DIR / "experiment.jsonl"
GOLD_DIR      = EXPERIMENT_DIR / "gold"; GOLD_DIR.mkdir(parents=True, exist_ok=True)
SLIM_PATH     = GOLD_DIR / "annotation_corpus.jsonl"
GOLD_PATH     = GOLD_DIR / f"manual_events.{VARIANT}.jsonl"
MANIFEST_PATH = GOLD_DIR / "annotation_progress.json"
EVAL_PATH     = GOLD_DIR / f"annotation_eval.{VARIANT}.yaml"

# For each annotation-ready summary, keep these fields to have meta data
KEEP_FIELDS = ("wikidata_id","summary_id","lang","text","sentences","n_sentences","events")
def uid_of(r): return f'{r["wikidata_id"]}__{r["summary_id"]}'   # composite key (summary_id is a lang code)

# Validate the important annotation metadata
print("config OK:", EXPERIMENT_DIR.name, "| variant:", VARIANT, "| annotator:", ANNOTATOR)

config OK: experiment_test_9385_20260609_1013 | variant: non_anon | annotator: bugra


### (Slim) corpus cache

**Goal:** Build a small, reusable copy of the corpus that keeps only the text fields annotation needs, so we do not reload the multi-GB `experiment.jsonl` (embeddings and baselines) on every session.

In [2]:
def build_slim_corpus(source_path: Path, dest_path: Path, keep_fields=KEEP_FIELDS) -> int:
    n_rows = 0
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".tmp")
    with source_path.open(encoding="utf-8") as infile, tmp_path.open("w", encoding="utf-8") as outfile:
        for line in infile:
            if not line.strip(): continue
            row = json.loads(line)
            outfile.write(json.dumps({field: row[field] for field in keep_fields}, ensure_ascii=False) + "\n")
            n_rows += 1
    tmp_path.replace(dest_path)
    return n_rows

# If the path doesn't exist, inform that it's written into cache for the first time. 
# For other runs, the same cache will be use to reduce computational overhead
if not SLIM_PATH.exists():
    print(f"Building slim cache from {CORPUS_PATH.name} (one-time, ~1-2 min)...")
    print(f"  wrote {build_slim_corpus(CORPUS_PATH, SLIM_PATH)} rows -> {SLIM_PATH.name}")

# Load the slim cache into memory and index it by composite uid for fast lookup.
corpus = [json.loads(line) for line in SLIM_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
by_uid = {uid_of(row): row for row in corpus}
print(f"loaded slim corpus: {len(corpus)} summaries ({len(by_uid)} unique uids)")

loaded slim corpus: 5666 summaries (5666 unique uids)


### Storage helpers

**Goal:** Define resumable, crash-safe persistence for the manual gold annotations: load existing gold annotations, save them atomically, track per-annotator progress in a JSON file, and build one gold record per summary.

In [3]:
# Load the saved gold annotations back into a {uid: record} dict (empty if none yet).
def load_gold(path: Path) -> dict:
    if not path.exists(): return {}
    records = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        record = json.loads(line)
        records[f'{record["wikidata_id"]}__{record["summary_id"]}'] = record
    return records

# Write the gold dict to disk atomically (temp file, then replace) so a crash can't corrupt it.
def save_gold(path: Path, gold: dict) -> None:
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as outfile:
        for record in gold.values():
            outfile.write(json.dumps(record, ensure_ascii=False) + "\n")
    tmp_path.replace(path)

# Track per-annotator progress (config + done uids) in a small manifest, so a later session can resume.
def update_manifest(path: Path, gold: dict) -> None:
    manifest = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    manifest.setdefault("config", {"seed": SEED, "min_sent": MIN_SENT, "max_sent": MAX_SENT})
    manifest.setdefault("annotators", {})[ANNOTATOR] = {
        "variant": VARIANT, "count": len(gold), "done_uids": sorted(gold.keys()),
    }
    path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

# Build one gold record for a summary from the annotator's events.
def gold_record(uid: str, events: list) -> dict:
    row = by_uid[uid]
    return {"annotator": ANNOTATOR, "variant": VARIANT,
            "wikidata_id": row["wikidata_id"], "summary_id": row["summary_id"],
            "events": events, "annotated_at": datetime.now().isoformat(timespec="seconds")}

print("storage helpers defined")

storage helpers defined


### Candidate selection

**Goal:** Choose which summaries to annotate: a fixed, seeded order of 5 to 10 sentence summaries, at most one per work for diversity, resuming past any already-annotated summaries so re-running continues where you left off.

In [4]:
def select_candidates(corpus, min_sent, max_sent, seed, restrict_to_ids=None, one_per_work=True):
    """Seeded order of candidate uids, filtered by sentence count.
    one_per_work=True keeps at most ONE summary per wikidata_id (distinct works) for diversity.
    restrict_to_ids: optional set of uids to keep (e.g. intersection with the anon corpus)."""
    eligible = [row for row in corpus if min_sent <= row["n_sentences"] <= max_sent]
    if restrict_to_ids is not None:
        keep_uids = set(restrict_to_ids); eligible = [row for row in eligible if uid_of(row) in keep_uids]
    uid_work_pairs = sorted((uid_of(row), row["wikidata_id"]) for row in eligible)   # stable base order before shuffle
    random.Random(seed).shuffle(uid_work_pairs)
    if one_per_work:
        seen_works, ordered_uids = set(), []
        for uid, work_id in uid_work_pairs:
            if work_id not in seen_works:
                seen_works.add(work_id); ordered_uids.append(uid)
        return ordered_uids
    return [uid for uid, _ in uid_work_pairs]

# Make sure each candidate is coming from a seperate work, has the size limitations
candidates = select_candidates(corpus, MIN_SENT, MAX_SENT, SEED)

# # uid -> record (resume from prior sessions) and determine the already-dones, thus the remaining summaries
gold = load_gold(GOLD_PATH)                
done = set(gold.keys())
remaining = [uid for uid in candidates if uid not in done]
n_works = len({by_uid[uid]["wikidata_id"] for uid in candidates})

# Print the perseverence-related information
print(f"{len(candidates)} candidates ({MIN_SENT}-{MAX_SENT} sentences, one per work; {n_works} distinct works) | "
      f"annotated {len(done)} | target N={N_TARGET} | next up: {len(remaining)} remaining")

1120 candidates (5-10 sentences, one per work; 1120 distinct works) | annotated 30 | target N=30 | next up: 1090 remaining


### MAVEN types + FrameNet guidance

**Goal:** Load the 168 MAVEN event types and build a FrameNet codebook to guide annotation: a definition per type plus a lemma-to-types lookup, so the annotator chooses types from the ontology rather than from the model's training signal. Both are cached to disk and reused.

In [5]:
from utils_maven import get_labels

# {type: definition|None}
FN_DEF_PATH = ROOT / "data" / "intermediate" / "framenet_definitions.json" 

# The 168 MAVEN event types, recovered from the BIO label set (drop the "B-"/"I-" prefix and "O").
def maven_types():
    return sorted({label[2:] for label in get_labels("") if label != "O"})

# Build (and cache) the FrameNet definition for each MAVEN type.
def build_framenet_definitions(types, def_path: Path):
    import nltk
    try:
        from nltk.corpus import framenet; framenet.frames()
    except LookupError:
        print("Downloading FrameNet (framenet_v17, one-time)...")
        nltk.download("framenet_v17", quiet=True)
        from nltk.corpus import framenet
    frame_by_name = {frame.name: frame for frame in framenet.frames()}
    clean_text = lambda s: re.sub(r"\s+", " ", re.sub(r"<[^>]+>", "", s)).strip()
    definitions = {}
    for type_name in types:
        frame = frame_by_name.get(type_name)
        definitions[type_name] = clean_text(frame.definition) if frame is not None else None
    def_path.parent.mkdir(parents=True, exist_ok=True)
    def_path.write_text(json.dumps(definitions, ensure_ascii=False), encoding="utf-8")
    return definitions

TYPES = maven_types(); TYPE_SET = set(TYPES)
if not FN_DEF_PATH.exists():
    build_framenet_definitions(TYPES, FN_DEF_PATH)
TYPE_DEF = json.loads(FN_DEF_PATH.read_text(encoding="utf-8"))    # {type: definition|None}

n_with_def = sum(1 for type_name in TYPES if TYPE_DEF.get(type_name))
print(f"{len(TYPES)} types | {n_with_def} with FrameNet definitions")

168 types | 144 with FrameNet definitions


### Blind annotation widget

**Goal:** The interactive UI for blind manual annotation. For each summary you pick a sentence, select the trigger word(s), choose a MAVEN event type (with FrameNet hints), and add the event. It never shows the model's predictions, and it autosaves after every change.

In [6]:
#Extract the words with pattern matching
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*")

# Split a sentence into word tokens with their character spans (punctuation excluded).
def tokenize(sentence):
    """[(idx, token_text, start, end), ...] — word spans excluding punctuation."""
    return [(i, match.group(0), match.start(), match.end()) for i, match in enumerate(WORD_RE.finditer(sentence))]

class Annotator:
    # Set up the widget over the given uids and jump to the first un-annotated summary.
    def __init__(self, uids):
        self.uids = uids
        self.cur = next((i for i, uid in enumerate(uids) if uid not in gold), 0)
        self.events = []                       # working events for the current summary
        self._build()
        self._load_current()
        display(self.ui)

    # Build all widgets and wire up their callbacks.
    def _build(self):
        self._syncing = False                  # reentrancy guard for word_sel <-> lookup sync
        self.progress = widgets.HTML()
        self.text     = widgets.HTML()
        self.sent_dd  = widgets.Dropdown(description="Sentence:", layout=widgets.Layout(width="95%"))
        self.word_sel = widgets.SelectMultiple(description="Trigger:", rows=6,
                                               layout=widgets.Layout(width="60%"))
        self.trig_lookup = widgets.Combobox(description="Lookup:", ensure_option=False, options=(),
                                            placeholder="type a sentence word, Enter",
                                            layout=widgets.Layout(width="60%"))
        # Type picker: a filter box + a SCROLLABLE listbox of all 168 types ("" = none).
        self.type_search = widgets.Text(description="Type:", placeholder="filter 168 types…",
                                        layout=widgets.Layout(width="60%"))
        self.type_list = widgets.Select(options=("",) + tuple(TYPES), value="", rows=10,
                                        description="", layout=widgets.Layout(width="60%"))
        self.type_hint = widgets.HTML()
        self.add_btn  = widgets.Button(description="Add event trigger and Event category",
                                       button_style="success", layout=widgets.Layout(width="320px"))
        self.events_box = widgets.VBox()
        self.prev_btn = widgets.Button(description="← Prev")
        self.next_btn = widgets.Button(description="Next →", button_style="primary")
        self.msg = widgets.HTML()

        self.sent_dd.observe(self._on_sent, names="value")
        self.word_sel.observe(self._on_words, names="value")
        self.trig_lookup.observe(self._on_lookup, names="value")
        self.type_search.observe(self._on_type_search, names="value")
        self.type_list.observe(self._on_type, names="value")
        self.add_btn.on_click(self._on_add)
        self.prev_btn.on_click(lambda button: self._go(-1))
        self.next_btn.on_click(lambda button: self._go(+1))
        self.ui = widgets.VBox([
            self.progress, self.text,
            self.sent_dd, self.word_sel,
            self.trig_lookup,
            self.type_search, self.type_list, self.type_hint,
            self.add_btn, widgets.HTML("<b>Events:</b>"), self.events_box,
            widgets.HBox([self.prev_btn, self.next_btn]), self.msg,
        ])

    # The full record of the summary currently being annotated.
    @property
    def row(self): return by_uid[self.uids[self.cur]]

    # Reset the type filter box and selection back to empty.
    def _reset_type(self):
        self.type_search.value = ""
        self.type_list.options = ("",) + tuple(TYPES)
        self.type_list.value = ""
        self.type_hint.value = ""

    # Load the current summary into the widgets (restoring any saved events for it).
    def _load_current(self):
        uid = self.uids[self.cur]
        self.events = [dict(event) for event in gold[uid]["events"]] if uid in gold else []
        sentences = self.row["sentences"]
        self.sent_dd.options = [(f"{i}: {sentence[:60]}", i) for i, sentence in enumerate(sentences)]
        self.sent_dd.value = 0
        self._on_sent(None); self._reset_type(); self._render()

    # Redraw the progress line, the sentence list, and the current event list.
    def _render(self):
        uid = self.uids[self.cur]
        self.progress.value = (f"<b>uid {self.cur+1}/{len(self.uids)}</b> &nbsp; {uid} &nbsp; "
                               f"(annotated {len(gold)}, target {N_TARGET})"
                               + ("  ✅ done" if uid in gold else ""))
        self.text.value = "<br>".join(f"<b>{i}.</b> {sentence}" for i, sentence in enumerate(self.row["sentences"]))
        event_rows = []
        for index, event in enumerate(self.events):
            label = widgets.HTML(f"s{event['sent_id']} &nbsp; <code>{event['trigger']}</code> "
                                 f"&rarr; <b>{event['event_type']}</b>")
            remove_button = widgets.Button(description="✕", layout=widgets.Layout(width="34px"))
            remove_button.on_click(lambda button, idx=index: self._del(idx))
            event_rows.append(widgets.HBox([remove_button, label]))
        self.events_box.children = event_rows

    # The surface string of the currently selected trigger word(s), or None.
    def _selected_surface(self):
        selected = sorted(self.word_sel.value)
        if self.sent_dd.value is None or not selected: return None
        tokens = tokenize(self.row["sentences"][self.sent_dd.value])
        return self.row["sentences"][self.sent_dd.value][tokens[selected[0]][2]:tokens[selected[-1]][3]]

    # Sentence changed: repopulate the word picker and lookup box for that sentence.
    def _on_sent(self, _):
        if self.sent_dd.value is None: return
        tokens = tokenize(self.row["sentences"][self.sent_dd.value])
        self._syncing = True
        self.word_sel.options = [(f"{i}: {token}", i) for i, token, start, end in tokens]
        self.word_sel.value = ()
        self.trig_lookup.options = tuple(dict.fromkeys(token for _, token, start, end in tokens))   # unique sentence words
        self.trig_lookup.value = ""
        self._syncing = False

    # Words selected: mirror the surface form into the lookup box.
    def _on_words(self, _):
        if self._syncing: return
        surface = self._selected_surface()
        if surface is None: return
        self._syncing = True; self.trig_lookup.value = surface; self._syncing = False

    # Word typed in the lookup box: select the matching sentence token.
    def _on_lookup(self, _):
        if self._syncing or self.sent_dd.value is None: return
        word = self.trig_lookup.value.lower().strip()
        tokens = tokenize(self.row["sentences"][self.sent_dd.value])
        match_indices = [i for i, token, start, end in tokens if token.lower() == word]
        if match_indices:                            # map typed word -> first matching token span
            self._syncing = True; self.word_sel.value = (match_indices[0],); self._syncing = False

    # Filter the 168-type list by the search box text.
    def _on_type_search(self, _):
        query = self.type_search.value.lower().strip()
        options = [type_name for type_name in TYPES if query in type_name.lower()] if query else list(TYPES)
        self.type_list.options = ("",) + tuple(options)
        self.type_list.value = ""

    # Show the FrameNet definition for the highlighted type.
    def _on_type(self, _):
        type_name = self.type_list.value
        if type_name in TYPE_SET:
            definition = TYPE_DEF.get(type_name) or "(no FrameNet definition — MAVEN-specific type; apply by judgment)"
            self.type_hint.value = f"<b>FrameNet Explanation:</b> <i>{definition}</i>"
        else:
            self.type_hint.value = ""

    # Add the selected trigger and type as an event, then autosave and redraw.
    def _on_add(self, _):
        selected = sorted(self.word_sel.value); event_type = self.type_list.value
        if not selected:   self.msg.value = "<span style='color:red'>select trigger word(s)</span>"; return
        if event_type not in TYPE_SET: self.msg.value = "<span style='color:red'>pick a valid type</span>"; return
        tokens = tokenize(self.row["sentences"][self.sent_dd.value])
        start = tokens[selected[0]][2]; end = tokens[selected[-1]][3]
        trigger = self.row["sentences"][self.sent_dd.value][start:end]
        self.events.append({"event_id": f"g{len(self.events)+1}", "sent_id": self.sent_dd.value,
                            "trigger": trigger, "event_type": event_type, "start": start, "end": end})
        self._syncing = True
        self.word_sel.value = (); self.trig_lookup.value = ""
        self._syncing = False
        self.msg.value = ""; self._reset_type()
        self._autosave(); self._render()

    # Remove an event by index, renumber the rest, then autosave and redraw.
    def _del(self, idx):
        self.events.pop(idx)
        for index, event in enumerate(self.events): event["event_id"] = f"g{index+1}"
        self._autosave(); self._render()

    # Persist only summaries with >=1 event; drop an emptied record.
    def _autosave(self):
        uid = self.uids[self.cur]
        if self.events:
            gold[uid] = gold_record(uid, [dict(event) for event in self.events])
        elif uid in gold:
            del gold[uid]
        save_gold(GOLD_PATH, gold); update_manifest(MANIFEST_PATH, gold)

    # Autosave the current summary, then move to the previous or next one.
    def _go(self, step):
        self._autosave()
        self.cur = max(0, min(len(self.uids) - 1, self.cur + step))
        self._load_current()

annotator = Annotator(candidates)

### Evaluation: manual gold vs BERT+CRF

**Goal:** Score the BERT+CRF extractor against the manual gold with the MAVEN protocol: match events per summary on sentence + span, then report micro precision/recall/F1 for trigger identification (span only) and classification (span and type), plus per-type and confusion views. Written to `gold/annotation_eval.<variant>.yaml`.

In [7]:
# === MAVEN-style evaluation: gold (manual) vs predicted (BERT+CRF) ===
# Match gold to predicted events per summary on sent_id + span; report micro P/R/F1 for trigger
# Identification (span only) and Classification (span AND type), plus per-type and a type-confusion
# view. Exact span = headline (MAVEN-style); overlap = lenient. Written to gold/annotation_eval.<variant>.yaml.

# One-to-one (gold_idx, pred_idx) span matches within a summary: same sent_id and (exact span, or best overlap).
def match_pairs(gold_events, pred_events, mode="exact"):
    used_pred, matches = set(), []
    for gold_idx, gold_event in enumerate(gold_events):
        best_match = None
        for pred_idx, pred_event in enumerate(pred_events):
            if pred_idx in used_pred or pred_event["sent_id"] != gold_event["sent_id"]: continue
            if mode == "exact":
                score = 1 if (pred_event["start"], pred_event["end"]) == (gold_event["start"], gold_event["end"]) else 0
            else:
                score = max(0, min(pred_event["end"], gold_event["end"]) - max(pred_event["start"], gold_event["start"]))
            if score > 0 and (best_match is None or score > best_match[0]): best_match = (score, pred_idx)
        if best_match: used_pred.add(best_match[1]); matches.append((gold_idx, best_match[1]))
    return matches

# Precision/recall/F1 from true positives, predicted count, and gold count.
def precision_recall_f1(true_positives, n_pred, n_gold):
    precision = true_positives / n_pred if n_pred else 0.0
    recall    = true_positives / n_gold if n_gold else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"P": round(precision, 4), "R": round(recall, 4), "F1": round(f1, 4)}

# Aggregate identification + classification metrics over all annotated summaries.
def evaluate(gold, by_uid, mode="exact"):
    identification_tp = classification_tp = n_pred = n_gold = 0
    type_tp, type_pred, type_gold = collections.Counter(), collections.Counter(), collections.Counter()
    confusion = collections.Counter(); n_no_gold = 0
    for uid, record in gold.items():
        gold_events = record["events"]; pred_events = by_uid.get(uid, {"events": []})["events"]
        if not gold_events: n_no_gold += 1
        n_gold += len(gold_events); n_pred += len(pred_events)
        for event in gold_events: type_gold[event["event_type"]] += 1
        for event in pred_events: type_pred[event["event_type"]] += 1
        for gold_idx, pred_idx in match_pairs(gold_events, pred_events, mode):
            identification_tp += 1
            gold_type, pred_type = gold_events[gold_idx]["event_type"], pred_events[pred_idx]["event_type"]
            if gold_type == pred_type: classification_tp += 1; type_tp[gold_type] += 1
            else: confusion[(gold_type, pred_type)] += 1
    per_type = {type_name: {"support": type_gold[type_name],
                            **precision_recall_f1(type_tp[type_name], type_pred[type_name], type_gold[type_name])}
                for type_name in sorted(type_gold)}
    return {"mode": mode, "n_summaries": len(gold), "n_summaries_no_gold_events": n_no_gold,
            "n_gold": n_gold, "n_pred": n_pred,
            "trigger_identification": precision_recall_f1(identification_tp, n_pred, n_gold),
            "trigger_classification": precision_recall_f1(classification_tp, n_pred, n_gold),
            "per_type": per_type,
            "top_confusions": [{"gold": gold_type, "pred": pred_type, "n": count}
                               for (gold_type, pred_type), count in confusion.most_common(15)]}

results = {mode: evaluate(gold, by_uid, mode=mode) for mode in ("exact", "overlap")}
EVAL_PATH.write_text(yaml.safe_dump(results, sort_keys=False, allow_unicode=True), encoding="utf-8")

for mode in ("exact", "overlap"):
    result = results[mode]
    print(f"[{mode}]  summaries={result['n_summaries']} (no-gold-events: {result['n_summaries_no_gold_events']})  "
          f"gold={result['n_gold']}  pred={result['n_pred']}")
    print(f"   Trigger ID : {result['trigger_identification']}")
    print(f"   Trigger CLS: {result['trigger_classification']}")
print(f"\nwrote {EVAL_PATH.name}")

[exact]  summaries=30 (no-gold-events: 0)  gold=431  pred=418
   Trigger ID : {'P': 0.7321, 'R': 0.71, 'F1': 0.7208}
   Trigger CLS: {'P': 0.5383, 'R': 0.522, 'F1': 0.53}
[overlap]  summaries=30 (no-gold-events: 0)  gold=431  pred=418
   Trigger ID : {'P': 0.7656, 'R': 0.7425, 'F1': 0.7538}
   Trigger CLS: {'P': 0.5478, 'R': 0.5313, 'F1': 0.5395}

wrote annotation_eval.non_anon.yaml


### Spot check: examples per error bucket

**Goal:** Spot-check the matches by sorting events into four buckets (correct, type-wrong, missed by the model, spurious model prediction) and showing a few random examples per bucket with the trigger marked. Read-only; reuses the matches from the evaluation cell.

In [8]:
import random
N_PER = 2           # Tweak to show N_PER random examples per bucket with the trigger marked
SEED_INSPECT = 0    # Tweak to get other random examplars

# Sort each matched/unmatched event into one of the four buckets, with its sentence + span for display.
def bucket_examples(gold, by_uid, mode="exact"):
    buckets = {"correct (span+type)": [], "span match, type WRONG": [],
               "missed by model (FN)": [], "spurious model pred (FP)": []}
    for uid, record in gold.items():
        gold_events = record["events"]; row = by_uid.get(uid, {"events": [], "sentences": []})
        pred_events, sentences = row["events"], row["sentences"]
        def sentence_at(i): return sentences[i] if i < len(sentences) else ""
        matches = match_pairs(gold_events, pred_events, mode)
        matched_gold = {gold_idx for gold_idx, _ in matches}; matched_pred = {pred_idx for _, pred_idx in matches}
        for gold_idx, pred_idx in matches:
            gold_event, pred_event = gold_events[gold_idx], pred_events[pred_idx]
            key = "correct (span+type)" if gold_event["event_type"] == pred_event["event_type"] else "span match, type WRONG"
            buckets[key].append({"uid": uid, "sent": sentence_at(gold_event["sent_id"]), "span": (gold_event["start"], gold_event["end"]),
                                 "g": (gold_event["trigger"], gold_event["event_type"]), "p": (pred_event["trigger"], pred_event["event_type"])})
        for gold_idx, gold_event in enumerate(gold_events):
            if gold_idx not in matched_gold:
                buckets["missed by model (FN)"].append({"uid": uid, "sent": sentence_at(gold_event["sent_id"]),
                    "span": (gold_event["start"], gold_event["end"]), "g": (gold_event["trigger"], gold_event["event_type"]), "p": None})
        for pred_idx, pred_event in enumerate(pred_events):
            if pred_idx not in matched_pred:
                buckets["spurious model pred (FP)"].append({"uid": uid, "sent": sentence_at(pred_event["sent_id"]),
                    "span": (pred_event["start"], pred_event["end"]), "g": None, "p": (pred_event["trigger"], pred_event["event_type"])})
    return buckets

# Wrap the trigger span in marker brackets for display.
def mark_span(sentence, span):
    start, end = span; return sentence[:start] + "【" + sentence[start:end] + "】" + sentence[end:]

buckets = bucket_examples(gold, by_uid, "exact")
rng = random.Random(SEED_INSPECT)
lines = ["## Gold vs BERT+CRF — random examples per bucket (exact-span)\n"]
for name, items in buckets.items():
    lines.append(f"### {name} — {len(items)} total")
    if not items:
        lines.append("_(none)_\n"); continue
    for example in rng.sample(items, min(N_PER, len(items))):
        lines.append(f"- `{example['uid']}` — {mark_span(example['sent'], example['span'])}")
        if example["g"]: lines.append(f"    - **gold**: `{example['g'][0]}` → **{example['g'][1]}**")
        if example["p"]: lines.append(f"    - **bert+crf**: `{example['p'][0]}` → **{example['p'][1]}**")
    lines.append("")
display(Markdown("\n".join(lines)))

## Gold vs BERT+CRF — random examples per bucket (exact-span)

### correct (span+type) — 225 total
- `1041839__it` — Monique encourages and 【trains】 him, teaching him how to ski the treacherous K12 piste.
    - **gold**: `trains` → **Education_teaching**
    - **bert+crf**: `trains` → **Education_teaching**
- `152105__de` — At the end of the film, Watanabe is dead; the politicians argue over who gets the credit for 【building】 the playground.
    - **gold**: `building` → **Building**
    - **bert+crf**: `building` → **Building**

### span match, type WRONG — 81 total
- `830773__es` — There the "king of the gangsters", Johnny Rocco (Edward G. Robinson), goes into hiding by becoming the master of the place and 【holding】 Nora Temple (Lauren Bacall), the owner of the hotel, her invalid father-in-law (Lionel Barrymore) and war veteran Frank McCloud (Humphrey Bogart) at gunpoint.
    - **gold**: `holding` → **Hold**
    - **bert+crf**: `holding` → **Defending**
- `2365679__fr` — But when she is mysteriously kidnapped, Jack 【realizes】 that his dark past is catching up with him.
    - **gold**: `realizes` → **Coming_to_believe**
    - **bert+crf**: `realizes` → **Know**

### missed by model (FN) — 125 total
- `682407__de` — The film ends with Michael's relatives dissolving the household some time after his death, opening the door to the basement room where Wolfgang was 【kept】 hidden.
    - **gold**: `kept` → **Hiding_objects**
- `2349200__it` — When she dies shortly afterwards as a result of an accidental fall, the boy thinks she was killed by her own husband and 【tries】 in every way to protect his idol, but his desperate and imaginative attempts only make the butler's position more difficult.
    - **gold**: `tries` → **Aiming**

### spurious model pred (FP) — 112 total
- `2570507__it` — However, Dementia, to spite her sister, reveals to Gustave the alternative that avoids 【death】, namely homework.
    - **bert+crf**: `death` → **Death**
- `2570507__it` — The story begins on a sailing ship that soon 【finds】 itself in the grip of a storm.
    - **bert+crf**: `finds` → **Know**


### Thesis table: extraction quality

**Goal:** Assemble the thesis table of BERT+CRF trigger-extraction quality (identification and classification, micro and macro, exact and overlap spans) next to the reported MAVEN reference, and save it to CSV.

In [9]:
# Macro-F1: unweighted mean of per-type F1 over types that have gold support.
def macro_f1(per_type):
    f1_scores = [stats["F1"] for stats in per_type.values() if stats["support"] > 0]
    return round(100 * sum(f1_scores) / len(f1_scores), 1) if f1_scores else None

def as_percent(value): return round(100 * value, 1)

table_rows = []
for mode in ("exact", "overlap"):
    result = results[mode]
    identification, classification = result["trigger_identification"], result["trigger_classification"]
    table_rows.append({"Dataset": "TMA", "Evaluation": "Trigger identification", "Span match": mode,
                 "P": as_percent(identification["P"]), "R": as_percent(identification["R"]), "F1": as_percent(identification["F1"]),
                 "Macro-F1": None, "n_gold": result["n_gold"], "n_pred": result["n_pred"]})
    table_rows.append({"Dataset": "TMA", "Evaluation": "Trigger classification", "Span match": mode,
                 "P": as_percent(classification["P"]), "R": as_percent(classification["R"]), "F1": as_percent(classification["F1"]),
                 "Macro-F1": macro_f1(result["per_type"]), "n_gold": result["n_gold"], "n_pred": result["n_pred"]})
table_rows.append({"Dataset": "MAVEN", "Evaluation": "Trigger classification", "Span match": "exact",
             "P": 65.0, "R": 70.9, "F1": 67.8, "Macro-F1": None, "n_gold": None, "n_pred": None})

columns = ["Dataset", "Evaluation", "Span match", "P", "R", "F1", "Macro-F1", "n_gold", "n_pred"]
thesis_table = pd.DataFrame(table_rows)[columns]

n_summaries = results["exact"]["n_summaries"]
print(f"BERT+CRF event-trigger extraction (micro, span-based; values in %; N_TMA={n_summaries} summaries)\n")
print(thesis_table.fillna("—").to_string(index=False))

out_csv = GOLD_DIR / f"thesis_extraction_metrics.{VARIANT}.csv"
thesis_table.to_csv(out_csv, index=False)
print(f"\nsaved -> {out_csv.name}")

BERT+CRF event-trigger extraction (micro, span-based; values in %; N_TMA=30 summaries)

Dataset             Evaluation Span match    P    R   F1 Macro-F1 n_gold n_pred
    TMA Trigger identification      exact 73.2 71.0 72.1        —  431.0  418.0
    TMA Trigger classification      exact 53.8 52.2 53.0     41.5  431.0  418.0
    TMA Trigger identification    overlap 76.6 74.2 75.4        —  431.0  418.0
    TMA Trigger classification    overlap 54.8 53.1 53.9     42.0  431.0  418.0
  MAVEN Trigger classification      exact 65.0 70.9 67.8        —      —      —

saved -> thesis_extraction_metrics.non_anon.csv
